use perturbation+cell type as the target of prediction, with primary cells as the part of the validation

# environment setup

In [1]:
!python --version

# prompt: check if a file "dummy.txt" exists in /content folder

import os


# Example usage:
flag_file_path = "/content/dummy.txt"
if os.path.exists(flag_file_path):
    print(f"environment set up complete.")
    import os, sys

    nb_path = "/content/content/notebookslib"

    sys.path.insert(0, nb_path)
    sys.path
    !ls $nb_path
else:
    print(f"setting up environment. Please wait")
    #
    import os, sys
    from google.colab import drive

    drive.mount("/content/drive")
    #
    !cp /content/drive/MyDrive/Colab\ Notebooks/notebookslib-2025.tar.gz /
    !tar -xzf /notebookslib-2025.tar.gz
    #
    import os, sys

    nb_path = "/content/content/notebookslib"

    sys.path.insert(0, nb_path)
    sys.path
    !ls $nb_path

    !rm -rf $nb_path/scvi*
    !rm -rf $nb_path/torch
    #!rm -rf $nb_path/torch-2.1.2*
    #!rm -rf $nb_path/jax*

    #!pip install torch==2.1.2 wandb louvain scvi_tools==1.1.6 torchvision jax jaxlib accelerate==1.1.0 # torch==2.3.0 torchvision==0.18.0
    !pip install torch==2.3.0 wandb louvain scvi_tools==1.1.6 torchvision torchtext accelerate==1.1.0 # torch==2.3.0 torchvision==0.18.0
    #!pip install --upgrade jax jaxlib
    !touch /content/dummy.txt

Python 3.11.11
environment set up complete.
absl						 numpy-2.1.3.dist-info
absl_py-2.1.0.dist-info				 numpy.libs
aiohappyeyeballs				 numpyro
aiohappyeyeballs-2.4.6.dist-info		 numpyro-0.17.0.dist-info
aiohttp						 nvidia
aiohttp-3.11.12.dist-info			 nvidia_cublas_cu12-12.1.3.1.dist-info
aiosignal					 nvidia_cuda_cupti_cu12-12.1.105.dist-info
aiosignal-1.3.2.dist-info			 nvidia_cuda_nvrtc_cu12-12.1.105.dist-info
anndata						 nvidia_cuda_runtime_cu12-12.1.105.dist-info
anndata-0.11.3.dist-info			 nvidia_cudnn_cu12-8.9.2.26.dist-info
array_api_compat				 nvidia_cufft_cu12-11.0.2.54.dist-info
array_api_compat-1.10.0.dist-info		 nvidia_curand_cu12-10.3.2.106.dist-info
attr						 nvidia_cusolver_cu12-11.4.5.107.dist-info
attrs						 nvidia_cusparse_cu12-12.1.0.106.dist-info
attrs-25.1.0.dist-info				 nvidia_nccl_cu12-2.20.5.dist-info
bin						 nvidia_nvjitlink_cu12-12.8.61.dist-info
cached_property-2.0.1.dist-info			 nvidia_nvtx_cu12-12.1.105.dist-info
cached_property.py				 openpyxl


**Need to restart environment here**

# importing

In [ ]:
import copy
import gc
import json
import os
from pathlib import Path
import sys
import time
import traceback
from typing import List, Tuple, Dict, Union, Optional
import warnings
import random

import torch
from anndata import AnnData
import scanpy as sc
import numpy as np

# import scvi
import wandb
from scipy.sparse import issparse
import matplotlib.pyplot as plt
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torchtext.vocab import Vocab
from torchtext._torchtext import (
    Vocab as VocabPybind,
)


: 

In [3]:
sys.path.insert(0, "../")
import scgpt as scg
from scgpt.model import TransformerModel, AdversarialDiscriminator
from scgpt.tokenizer import tokenize_and_pad_batch, random_mask_value
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.loss import (
    masked_mse_loss,
    masked_relative_error,
    criterion_neg_log_bernoulli,
)
from scgpt.preprocess import Preprocessor
from scgpt import SubsetsBatchSampler
from scgpt.utils import set_seed, eval_scib_metrics, load_pretrained

sc.set_figure_params(figsize=(4, 4))
os.environ["KMP_WARNINGS"] = "off"
warnings.filterwarnings("ignore")

/content/content/notebookslib/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/content/content/notebookslib/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


In [4]:
import scvi

# Step 2: use pertTF github

In [5]:
!git clone https://github.com/davidliwei/pertTF.git

Cloning into 'pertTF'...
remote: Enumerating objects: 194, done.
remote: Counting objects: 100% (194/194), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 194 (delta 110), reused 90 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (194/194), 48.81 KiB | 757.00 KiB/s, done.
Resolving deltas: 100% (110/110), done.


In [6]:
sys.path.insert(0, "/content/pertTF/")

from perttf.model.pertTF import PerturbationTFModel
from perttf.model.config_gen import generate_config
from perttf.model.train_data_gen import produce_training_datasets
from perttf.model.train_function import train, wrapper_train, eval_testdata

# Step 3: hyperparameter set up

In [7]:
hyperparameter_defaults = dict(
    seed=42,
    # dataset_name="PBMC_10K", # Dataset name
    dataset_name="pancreatic",
    do_train=True,  # Flag to indicate whether to do update model parameters during training
    # load_model="/content/drive/MyDrive/Colab Notebooks/scGPT/pretrain_blood", # Path to pre-trained model
    load_model=None,
    GEPC=True,  # Gene expression modelling for cell objective
    ecs_thres=0.7,  # Elastic cell similarity objective, 0.0 to 1.0, 0.0 to disable
    dab_weight=0.0,  # 2000.0, # DAR objective weight for batch correction; if has batch, set to 1.0
    this_weight=1.0,  # weight for predicting the expression of current cell
    next_weight=00.0,  # weight for predicting the next cell
    n_rounds=1,  # number of rounds for generating the next cell
    next_cell_pred_type="identity",  # the method to predict the next cell, either "pert" (random next cell within the same cell type) or "identity" (the same cell). If "identity", set next_weight=0
    #
    ecs_weight=1.0,  # weight for predicting the similarity of cells
    cell_type_classifier=True,  #  do we need the trasnformer to separate cell types?
    cell_type_classifier_weight=1.0,
    perturbation_classifier_weight=10.0,
    perturbation_input=False,  # use perturbation as input?
    CCE=False,  # Contrastive cell embedding objective
    mask_ratio=0.15,  # Default mask ratio
    epochs=100,  # Default number of epochs for fine-tuning
    n_bins=51,  # Default number of bins for value binning in data pre-processing
    # lr=1e-4, # Default learning rate for fine-tuning
    lr=1e-3,  # learning rate for learning de novo
    batch_size=32,  # Default batch size for fine-tuning
    layer_size=32,  # defalut 32
    nlayers=2,
    nhead=4,  # if load model, batch_size, layer_size, nlayers, nhead will be ignored
    dropout=0.4,  # Default dropout rate during model fine-tuning
    schedule_ratio=0.99,  # Default rate for learning rate decay
    save_eval_interval=5,  # Default model evaluation interval
    log_interval=100,  # Default log interval
    fast_transformer=True,  # Default setting
    pre_norm=False,  # Default setting
    amp=True,  # # Default setting: Automatic Mixed Precision
    do_sample_in_train=False,  # sample the bernoulli in training
    ADV=False,  # Adversarial training for batch correction
    adv_weight=10000,
    adv_E_delay_epochs=2,  # delay adversarial training on encoder for a few epochs
    adv_D_delay_epochs=2,
    lr_ADV=1e-3,  # learning rate for discriminator, used when ADV is True
    DSBN=False,  # True if (config.dab_weight >0 or config.ADV ) else False  # Domain-spec batchnorm; default is True
    per_seq_batch_sample=False,  # DSBN # default True
    use_batch_label=False,  # default: equal to DSBN
    schedule_interval=1,
    explicit_zero_prob=True,  # whether explicit bernoulli for zeros
    n_hvg=3000,  # number of highly variable genes
    mask_value=-1,
    pad_value=-2,
    pad_token="<pad>",
)


wandb configuration

In [8]:
os.environ["WANDB_API_KEY"] = "ebbd6ca0c8ce4d7052af32e9bd74fa8fc64a2a3e"


In [9]:
config, run_session = generate_config(hyperparameter_defaults, wandb_mode="online")


wandb: Currently logged in as: li-david-wei (li-david-wei-children-s-national-hospital) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


{'seed': 42, 'dataset_name': 'pancreatic', 'do_train': True, 'load_model': None, 'GEPC': True, 'ecs_thres': 0.7, 'dab_weight': 0.0, 'this_weight': 1.0, 'next_weight': 0.0, 'n_rounds': 1, 'next_cell_pred_type': 'identity', 'ecs_weight': 1.0, 'cell_type_classifier': True, 'cell_type_classifier_weight': 1.0, 'perturbation_classifier_weight': 10.0, 'perturbation_input': False, 'CCE': False, 'mask_ratio': 0.15, 'epochs': 100, 'n_bins': 51, 'lr': 0.001, 'batch_size': 32, 'layer_size': 32, 'nlayers': 2, 'nhead': 4, 'dropout': 0.4, 'schedule_ratio': 0.99, 'save_eval_interval': 5, 'log_interval': 100, 'fast_transformer': True, 'pre_norm': False, 'amp': True, 'do_sample_in_train': False, 'ADV': False, 'adv_weight': 10000, 'adv_E_delay_epochs': 2, 'adv_D_delay_epochs': 2, 'lr_ADV': 0.001, 'DSBN': False, 'per_seq_batch_sample': False, 'use_batch_label': False, 'schedule_interval': 1, 'explicit_zero_prob': True, 'n_hvg': 3000, 'mask_value': -1, 'pad_value': -2, 'pad_token': '<pad>', 'special_tokens

In [10]:
# the following parameters have been moved to config
# DSBN =False # True if (config.dab_weight >0 or config.ADV ) else False  # Domain-spec batchnorm; default is True
# per_seq_batch_sample =  False # DSBN # default True


dataset_name = config.dataset_name
save_dir = Path(f"./save/dev_{dataset_name}-{time.strftime('%b%d-%H-%M')}/")
save_dir.mkdir(parents=True, exist_ok=True)
print(f"save to {save_dir}")
logger = scg.logger
scg.utils.add_file_handler(logger, save_dir / "run.log")

save to save/dev_pancreatic-Feb25-15-48


In [11]:
# embsize = config.layer_size
# nhead = config.nhead
# nlayers = config.nlayers
# d_hid = config.layer_size

# step 4: load scrna-seq dataset

In [12]:
adata0 = sc.read_h5ad(
    "/content/drive/MyDrive/Colab Notebooks/scGPT/pancreatic/data/D18_diabetes_merged.h5ad"
)
# adata0=sc.read_h5ad("/content/drive/MyDrive/Colab Notebooks/scGPT/pancreatic/data/D18_diabetes_merged_reduced.h5ad")
adata0.layers["GPTin"] = adata0.X.copy()


In [13]:
# set up the preprocessor, use the args to config the workflow


preprocessor = Preprocessor(
    use_key="GPTin",  # the key in adata.layers to use as raw data
    # filter_gene_by_counts=3,  # step 1
    # filter_cell_by_counts=False,  # step 2
    normalize_total=None,  # 3. whether to normalize the raw data and to what sum
    # result_normed_key="X_normed",  # the key in adata.layers to store the normalized data
    log1p=False,  # 4. whether to log1p the normalized data
    # result_log1p_key="X_log1p",
    subset_hvg=False,  # 5. whether to subset the raw data to highly variable genes; use n_hvg default
    hvg_flavor="seurat_v3",  # if data_is_raw else "cell_ranger",
    binning=config.n_bins,  # 6. whether to bin the raw data and to what number of bins
    # binning=0,  # 6. whether to bin the raw data and to what number of bins
    result_binned_key="X_binned",  # the key in adata.layers to store the binned data
)
preprocessor(adata0, batch_key=None)


scGPT - INFO - Binning data ...


In [14]:
# this is for integration training
if False:
    adata = adata0[
        (adata0.obs["batch"] == 0) | (adata0.obs["Diabetes_status"] == "Non-diabetic"),
        :,
    ]

    # remove unknown cell type
    adata = adata[adata.obs["celltype"] != "unknown", :]
    # only test cells where two batches share the same cell type

    adata = adata[adata.obs["celltype"].isin(["SC-beta", "SC-alpha", "SC-delta"]), :]


if True:
    adata0.obs.loc[adata0.obs["genotype"].isna(), "genotype"] = "WT"
    adata = adata0[adata0.obs["batch"] == 0, :]  # only use cell line data for training
    adata_prim = adata0[
        adata0.obs["batch"] == 1, :
    ]  # a separate primary data as a validation data
    # prompt: randomly select 1000 n_obs from adata_prim

    import random

    random_indices = random.sample(range(adata_prim.n_obs), min(3000, adata_prim.n_obs))
    adata_prim = adata_prim[random_indices]
    adata_prim
adata

View of AnnData object with n_obs × n_vars = 31293 × 3354
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'gene', 'sgrna', 'umi_count', 'nCount_sgRNA', 'nFeature_sgRNA', 'nCount_sgRNA_guides', 'nFeature_sgRNA_guides', 'percent.mt', 'RNA_snn_res.0.5', 'seurat_clusters', 'S.Score', 'G2M.Score', 'Phase', 'old.ident', 'celltype', 'genotype', 'sub.cluster', 'celltype_1', 'celltype.genotype', 'celltype_2', 'SC.alpha.PS', 'Ductal.PS', 'Endocrine.precursor.PS', 'Liver.PS', 'SC.beta.PS', 'SC.delta.PS', 'SC.EC.PS', 'Stromal.PS', 'percent_mito', 'n_counts', 'log10_n_counts', 'log_n_counts', 'n_genes', 'log10_n_genes', 'diabetes_batch', 'donor', 'leiden', 'Diabetes_status', 'batch'
    obsm: 'X_pca', 'X_umap', 'bin_edges'
    layers: 'GPTin', 'X_binned'

In [15]:
adata_prim

View of AnnData object with n_obs × n_vars = 3000 × 3354
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'gene', 'sgrna', 'umi_count', 'nCount_sgRNA', 'nFeature_sgRNA', 'nCount_sgRNA_guides', 'nFeature_sgRNA_guides', 'percent.mt', 'RNA_snn_res.0.5', 'seurat_clusters', 'S.Score', 'G2M.Score', 'Phase', 'old.ident', 'celltype', 'genotype', 'sub.cluster', 'celltype_1', 'celltype.genotype', 'celltype_2', 'SC.alpha.PS', 'Ductal.PS', 'Endocrine.precursor.PS', 'Liver.PS', 'SC.beta.PS', 'SC.delta.PS', 'SC.EC.PS', 'Stromal.PS', 'percent_mito', 'n_counts', 'log10_n_counts', 'log_n_counts', 'n_genes', 'log10_n_genes', 'diabetes_batch', 'donor', 'leiden', 'Diabetes_status', 'batch'
    obsm: 'X_pca', 'X_umap', 'bin_edges'
    layers: 'GPTin', 'X_binned'

In [16]:
adata.obs.Diabetes_status.value_counts(dropna=False)

,count
Diabetes_status,
NaN,31293


In [17]:
adata.obs["genotype"].value_counts(dropna=False)

,count
genotype,
PDX1,4151
WT,4086
NEUROD1,2715
FOXA1,2629
MNX1,2294
NKX6-1,1838
TLE3,1791
PROSER1,1151
NKX2-2,948


In [18]:
adata.obs["genotype"].value_counts(dropna=False)

,count
genotype,
PDX1,4151
WT,4086
NEUROD1,2715
FOXA1,2629
MNX1,2294
NKX6-1,1838
TLE3,1791
PROSER1,1151
NKX2-2,948


In [19]:
adata.obs

,orig.ident,nCount_RNA,nFeature_RNA,gene,sgrna,umi_count,nCount_sgRNA,nFeature_sgRNA,nCount_sgRNA_guides,nFeature_sgRNA_guides,...,n_counts,log10_n_counts,log_n_counts,n_genes,log10_n_genes,diabetes_batch,donor,leiden,Diabetes_status,batch
EC1_AAACCCAAGAAGTCCG-1,Sample_L_1_3DEC,12089.0,4199.0,75_TLE3,75_TLE3,79.0,79.0,1.0,79.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
EC1_AAACCCAAGATAGTGT-1,Sample_L_1_3DEC,6601.0,2953.0,27_GATA4,27_GATA4,55.0,55.0,1.0,55.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
EC1_AAACCCACACCTCGTT-1,Sample_L_1_3DEC,11164.0,3944.0,110_WT,110_WT,399.0,399.0,1.0,399.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
EC1_AAACCCACACGGCGTT-1,Sample_L_1_3DEC,8629.0,3534.0,103_MNX1,103_MNX1,63.0,63.0,1.0,63.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
EC1_AAACCCACAGCTTCGG-1,Sample_L_1_3DEC,6410.0,2949.0,91_NKX6-1,91_NKX6-1,10.0,10.0,1.0,10.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
EC2_TTTGTTGTCACTTGGA-1,Sample_L_2_3DEC,16919.0,5791.0,112_PDX1,112_PDX1,28.0,28.0,1.0,28.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
EC2_TTTGTTGTCCTGTAAG-1,Sample_L_2_3DEC,11894.0,4354.0,91_NKX6-1,91_NKX6-1,351.0,351.0,1.0,351.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
EC2_TTTGTTGTCTCCATAT-1,Sample_L_2_3DEC,6973.0,3238.0,115_PDX1,115_PDX1,54.0,54.0,1.0,54.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
EC2_TTTGTTGTCTGATGGT-1,Sample_L_2_3DEC,8516.0,3635.0,75_TLE3,75_TLE3,46.0,46.0,1.0,46.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [20]:
# for debugging only; use top 2k genes only
if False:
    value_x = raw_data_f[3, :].toarray()
    value_y = adata.X[3, :]
    # prompt: calculate and plot the correlation between value_x and value_y

    import numpy as np
    import matplotlib.pyplot as plt

    # Assuming value_x and value_y are defined as in the provided code
    # Replace with your actual data if needed

    correlation = np.corrcoef(value_x, value_y)[0, 1]
    print(f"Correlation between value_x and value_y: {correlation}")

    plt.scatter(value_x, value_y)
    plt.xlabel("value_x")
    plt.ylabel("value_y")
    plt.title(f"Correlation: {correlation:.2f}")
    plt.show()

In [21]:
set(adata.obs["genotype"])

{'ARX',
 'BCOR',
 'BMPR1A',
 'FOXA1',
 'FOXA2',
 'GATA4',
 'GATA4het',
 'GATA6',
 'GATA6het',
 'GLIS3',
 'GSC',
 'HHEX',
 'HHEXe',
 'HHEXhet',
 'HNF4A',
 'KDM2B',
 'MNX1',
 'NANOGe',
 'NEUROD1',
 'NGN3',
 'NKX2-2',
 'NKX6-1',
 'ONECUT1e',
 'OTUD5',
 'PAX6',
 'PBX1',
 'PDX1',
 'PDX1het',
 'PROSER1',
 'QSER1',
 'QSER1TET1',
 'RFX6',
 'TADA2B',
 'TET1',
 'TET1/2/3',
 'TLE3',
 'UBA6',
 'WT'}

In [22]:
adata.obs["celltype"].value_counts(dropna=False)

,count
celltype,
Ductal,10539
SC-EC,6380
SC-alpha,4343
SC-beta,4174
Endocrine precursor,2933
Stromal,1825
SC-delta,816
Liver,283


In [23]:
adata.obs["genotype"].value_counts(dropna=False)

,count
genotype,
PDX1,4151
WT,4086
NEUROD1,2715
FOXA1,2629
MNX1,2294
NKX6-1,1838
TLE3,1791
PROSER1,1151
NKX2-2,948


In [24]:
adata.obs.loc[adata.obs["batch"] == 0, "celltype"].value_counts(dropna=False)

,count
celltype,
Ductal,10539
SC-EC,6380
SC-alpha,4343
SC-beta,4174
Endocrine precursor,2933
Stromal,1825
SC-delta,816
Liver,283


In [25]:
adata.obs.loc[adata.obs["batch"] == 1, "celltype"].value_counts(dropna=False)

,count
celltype,
Ductal,0
Endocrine precursor,0
Liver,0
SC-EC,0
SC-alpha,0
SC-beta,0
SC-delta,0
Stromal,0


In [26]:
adata.obs.loc[adata.obs["batch"] == 0, "genotype"].value_counts(dropna=False)

,count
genotype,
PDX1,4151
WT,4086
NEUROD1,2715
FOXA1,2629
MNX1,2294
NKX6-1,1838
TLE3,1791
PROSER1,1151
NKX2-2,948


In [27]:
adata.obs.loc[adata.obs["batch"] == 1, "genotype"].value_counts(dropna=False)

,count
genotype,
ARX,0
BCOR,0
BMPR1A,0
FOXA1,0
FOXA2,0
GATA4,0
GATA4het,0
GATA6,0
GATA6het,0


# step 5: preprocess dataset

In [28]:
# process the batch separately
# not needed, because the normalization is for each cell
# adata_batch0.layers['X_binned'][3,:].max()#
# adata=adata_batch0.concatenate(adata_batch1)

# adata.layers['GPTin'].toarray()

In [29]:
# adata.obs['batch_id'].value_counts(dropna=False)

Tokenize the input data for model fine-tuning

In [30]:
data_produced = produce_training_datasets(adata, config)

# cell_type_to_index=data_produced['cell_type_to_index']
# genotype_to_index=data_produced['genotype_to_index']


# num_batch_types=data_produced['num_batch_types']
# n_perturb=data_produced['n_perturb']
# n_cls=data_produced['n_cls']

scGPT - INFO - train set number of samples: 28163, 
	 feature length: 3001
scGPT - INFO - valid set number of samples: 3130, 
	 feature length: 3001


save data

In [31]:
# adata.obs['is_in_training']=adata.obs.index.isin(data_produced['cell_ids_train'])
adata0.obs["is_in_training"] = adata0.obs.index.isin(data_produced["cell_ids_train"])
# sum(adata.obs['is_in_training'])
adata0.write_h5ad(save_dir / f"adata_original_full.h5ad")

In [32]:
# train_data.shape

 # Step 6: Build or Load the pre-trained scGPT model

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab = data_produced["vocab"]

ntokens = len(vocab)  # size of vocabulary
model = PerturbationTFModel(
    data_produced["n_perturb"],  # n_perturb, # number of perturbation labels
    3,  # layers
    ntokens,
    config.layer_size,  # embsize,
    config.nhead,  # nhead,
    config.layer_size,  # d_hid,
    config.nlayers,  # nlayers,
    vocab=vocab,
    dropout=config.dropout,
    pad_token=config.pad_token,
    pad_value=config.pad_value,
    do_mvc=config.GEPC,
    do_dab=True if config.dab_weight > 0 else False,
    use_batch_labels=config.use_batch_label,
    num_batch_labels=data_produced["num_batch_types"],  # num_batch_types,
    domain_spec_batchnorm=config.DSBN,
    n_input_bins=config.n_bins,
    ecs_threshold=config.ecs_thres,
    explicit_zero_prob=config.explicit_zero_prob,
    use_fast_transformer=config.fast_transformer,
    pre_norm=config.pre_norm,
    n_cls=data_produced["n_cls"],  # n_cls, # number of cell type labels
    nlayers_cls=3,
)


In [34]:
if config.load_model is not None:
    load_pretrained(model, torch.load(model_file), verbose=False)

model.to(device)
wandb.watch(model)

In [35]:
print(model)


PerturbationTFModel(
  (encoder): GeneEncoder(
    (embedding): Embedding(3357, 32, padding_idx=3354)
    (enc_norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  )
  (value_encoder): ContinuousValueEncoder(
    (dropout): Dropout(p=0.4, inplace=False)
    (linear1): Linear(in_features=1, out_features=32, bias=True)
    (activation): ReLU()
    (linear2): Linear(in_features=32, out_features=32, bias=True)
    (norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=32, bias=True)
        (dropout): Dropout(p=0.4, inplace=False)
        (linear2): Linear(in_features=32, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affi

# Step 7: train

In [36]:
if config.per_seq_batch_sample:
    adata_prim_v = adata_prim[adata_prim.obs["batch_id"].argsort()].copy()
else:
    adata_prim_v = adata_prim.copy()
eval_adata_dict = {
    "validation": data_produced["adata_sorted"],
    "primary": adata_prim_v,
}


In [37]:
best_model = wrapper_train(
    model, config, data_produced, eval_adata_dict=eval_adata_dict
)

Output hidden; open in https://colab.research.google.com to view.

In [38]:
if False:
    ret_adata = results["adata"]
    ret_adata.obsm["X_pert_pred"].shape
    genotype_to_index

    import numpy as np

    # Assuming ret_adata.obsm['X_pert_pred'] is a numpy array or can be converted to one
    logits = ret_adata.obsm["X_pert_pred"]

    # Convert logits to probabilities using softmax
    X_pert_pred_probs = np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)

    # Assign the probabilities back to the AnnData object
    ret_adata.obsm["X_pert_pred_probs"] = X_pert_pred_probs

    # prompt: convert X_pert_pred_probs, which is the probabilities of each label, into label predictions, whose order is defined in genotype_to_index

    # Convert probabilities to predicted labels
    label_predictions = np.argmax(X_pert_pred_probs, axis=1)

    # Map predicted indices back to genotypes using genotype_to_index
    # Assuming genotype_to_index is a dictionary where keys are indices and values are genotypes
    index_to_genotype = {v: k for k, v in genotype_to_index.items()}
    predicted_genotypes = [index_to_genotype[i] for i in label_predictions]

    # Add the predicted genotypes to the AnnData object
    ret_adata.obs["predicted_genotype"] = predicted_genotypes

In [39]:
# ret_adata=results_p['adata']
# ret_adata.obs[['Diabetes_status','predicted_genotype']].value_counts(dropna=False)

In [40]:
# ret_adata.obs

In [41]:
# sc.pl.umap( ret_adata,color=["celltype"])

In [42]:
# sc.pl.umap( ret_adata,color=["predicted_genotype"])

In [43]:
# adata_prim.obs['celltype'].isin(cell_type_to_index)

In [44]:
# adata_prim.obs['genotype'].value_counts(dropna=False)

In [45]:
# adata_prim.obs.columns

In [46]:
artifact = wandb.Artifact(f"best_model", type="model")
glob_str = os.path.join(save_dir, "best_model.pt")
artifact.add_file(glob_str)
run_session.log_artifact(artifact)

run_session.finish()
wandb.finish()
gc.collect()

ValueError: Path is not a file: 'save/dev_pancreatic-Feb25-15-48/best_model.pt'

In [ ]:
# save the best model
# already incorporated into wrapper_train
if False:
    torch.save(best_model.state_dict(), save_dir / "best_model.pt")
    torch.save(vocab, save_dir / "vocab.pt")
    running_parameters = {
        "cell_type_to_index": data_produced["cell_type_to_index"],
        "genotype_to_index": data_produced["genotype_to_index"],
        "genes": data_produced["genes"],  # genes,
        "gene_ids": data_produced["gene_ids"],  # gene_ids,
    }
    running_parameters

    torch.save(running_parameters, save_dir / "running_parameters.pt")

# save the dataset
# torch.save(adata, save_dir / "adata.pt")
# torch.save(tokenized_train, save_dir / "tokenized_train.pt")
# torch.save(tokenized_valid, save_dir / "tokenized_valid.pt")
# torch.save(tokenized_train_next, save_dir / "tokenized_train_next.pt")
# torch.save(tokenized_valid_next, save_dir / "tokenized_valid_next.pt")


In [ ]:
# results['celltype_umap'].savefig(save_dir / f"test.png", dpi=300,bbox_inches='tight')

In [ ]:
!cp -r $save_dir /content/drive/MyDrive/Colab\ Notebooks/scGPT/pancreatic/